# CockroachDB Chat Store


`CockroachDBChatStore` persists `ChatMessage` history per session in CockroachDB. Each key maps to a single `JSONB` cell containing a JSON array of messages. (CockroachDB does not yet support `ARRAY(JSON)` as a column type, so messages live inside one JSONB array rather than as separate array elements.) Atomic appends use the JSONB `||` operator.


## Setup

In [ ]:
%pip install llama-index llama-index-cockroachdb llama-index-llms-openai

Start a local insecure cluster (development only):

```bash
cockroach start-single-node --insecure --listen-addr=localhost:26257
```


In [ ]:
import os
os.environ.setdefault("OPENAI_API_KEY", "sk-...")


## Build the store

In [ ]:
from llama_index.storage.chat_store.cockroachdb import CockroachDBChatStore

chat_store = CockroachDBChatStore.from_params(
    host="localhost", port=26257, database="defaultdb",
    user="root", password="",
    sslmode="disable",
    table_name="chat_history",
)

## Use as ChatMemoryBuffer backing

In [ ]:
from llama_index.core.memory import ChatMemoryBuffer

memory = ChatMemoryBuffer.from_defaults(
    token_limit=3000,
    chat_store=chat_store,
    chat_store_key="user_42",
)

## Plug into an agent or chat engine

In [ ]:
from llama_index.llms.openai import OpenAI
from llama_index.core.agent.workflow import FunctionAgent

def add(a: int, b: int) -> int:
    return a + b

agent = FunctionAgent(
    tools=[add],
    llm=OpenAI(model="gpt-4o-mini"),
)
response = await agent.run("what is 3 + 4?", memory=memory)
print(response)

## Persistence across sessions

Build a new chat store pointing at the same table and key to resume a conversation.

In [ ]:
next_session = CockroachDBChatStore.from_params(
    host="localhost", port=26257, database="defaultdb",
    user="root", password="", sslmode="disable",
    table_name="chat_history",
)
for msg in next_session.get_messages("user_42"):
    print(msg.role, msg.content[:80])

## Manual message management

In [ ]:
from llama_index.core.llms import ChatMessage, MessageRole

chat_store.set_messages(
    "sess-demo",
    [
        ChatMessage(role=MessageRole.USER, content="hi"),
        ChatMessage(role=MessageRole.ASSISTANT, content="hello"),
    ],
)

chat_store.add_message("sess-demo", ChatMessage(role=MessageRole.USER, content="again"))
for m in chat_store.get_messages("sess-demo"):
    print(m.role, m.content)

removed = chat_store.delete_last_message("sess-demo")
print("removed:", removed.content)

chat_store.delete_messages("sess-demo")

## Async

All operations have async counterparts (`aset_messages`, `async_add_message`, `aget_messages`, ...)

In [ ]:
await chat_store.aset_messages(
    "sess-async",
    [ChatMessage(role=MessageRole.USER, content="hi")],
)
print(await chat_store.aget_messages("sess-async"))